In [22]:
### Basic deep agent
import os,uuid
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

## StateBackend

In [19]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend,StoreBackend
from langchain.chat_models import init_chat_model
from langgraph.store.memory import InMemoryStore

In [5]:
model=init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A120F37710>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A121081110>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [32]:
llm=init_chat_model("groq:qwen/qwen3-32b")

In [6]:
agent=create_deep_agent(model=model)

In [ ]:
agent2=create_deep_agent(
        model=model, 
        backend=StateBackend(),
    )

In [9]:
result=agent2.invoke({
    "messages":[{
        "role":"user",
        "content":(
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2.Edit video\n3.Upload video\n"
            "Then tell me you've done it"
        )
    }]
})

In [10]:
print(result["messages"][-1].content)

File /notes/todo.txt has been created with the requested content.


In [11]:
result

{'messages': [HumanMessage(content="Create a file at /notes/todo.txt with exactly this content:\n1. Record video\n2.Edit video\n3.Upload video\nThen tell me you've done it", additional_kwargs={}, response_metadata={}, id='604ab468-2fc9-4e26-a3a1-f0a0ca002d99'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to create a file at /notes/todo.txt with content lines:\n\n1. Record video\n2.Edit video\n3.Upload video\n\nThen tell them done.\n\nWe need to use write_file tool. Path: /notes/todo.txt. Content: exactly as specified. Ensure newline after each line? The content lines should be exactly as shown, each line separated by newline. The user didn't specify trailing newline but typical. We'll write with newline after each line. Then respond that done.\n\nWe must not add any extra content. Just respond with confirmation.\n\nWe don't need to use write_todos.\n\nLet's call write_file.", 'tool_calls': [{'id': 'fc_82d47d88-94b5-4e91-a88e-d3bb684c5eab', 'function'

In [12]:
followup=agent2.invoke({
    "messages":result["messages"]+[{
        "role":"user",
        "content":"Read /notes/todo.txt back to me."
    }],
    "files":result.get("files",{})
})

print(followup["messages"][-1].content)

1. Record video  
2. Edit video  
3. Upload video


### FilesysytemBackend(local disk)

In [13]:
from deepagents.backends import FilesystemBackend
ROOT="."

agent3=create_deep_agent(
        model=model, 
        backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
    )

In [14]:
result2=agent3.invoke({
    "messages":[{
        "role":"user",
        "content":(
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2.Edit video\n3.Upload video\n"
            "Then tell me you've done it"
        )
    }]
})

In [15]:
print(result2["messages"][-1].content)

File created at /notes/todo.txt.


In [17]:
fresh_agent=create_deep_agent(
    model=model, 
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

In [18]:
followup=fresh_agent.invoke({
    "messages":result["messages"]+[{
        "role":"user",
        "content":"Read /notes/todo.txt back to me."
    }],
    "files":result.get("files",{})
})

print(followup["messages"][-1].content)

1. Record video
2.Edit video
3.Upload video


### Deep Agent StoreBackend verification

A Creates a deep agent backed by a LangGraph store, invokes it to write a file on one thread, then proves the backend works by reading that file back on a DIFFERENT thread - something StateBackend cannot do. """

In [20]:
store=InMemoryStore()

In [35]:
agent_with_thread=create_deep_agent(
    model=model,
    backend=StoreBackend(
        #local dev: static namespace. No delpoyment runtime needed.
        #In a langsmith Deployment you'd use the user-identity version
        namespace=lambda rt: ("demo-user",),
    ),
    store=store
)
print("agent successfully created with StoreBackend + static namespace")

agent successfully created with StoreBackend + static namespace


In [44]:
thread_1={"configurable":{"thread_id":str(uuid.uuid4())}}

result=agent_with_thread.invoke({
    "messages":[{
        "role":"user",
        "content":(
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2.Edit video\n3.Upload video\n"
            "Then tell me you've done it"
        )
    }]
},
config=thread_1
)

In [45]:
print(result["messages"][-1].content)

I’ve verified that /notes/todo.txt already contains the requested content. It’s ready.


In [46]:
thread_2={"configurable":{"thread_id":str(uuid.uuid4())}}

followp=agent_with_thread.invoke({
    "messages":[{
        "role":"user",
        "content":"Read /notes/todo.txt back to me."
    }]
},
config=thread_2
)

In [47]:
print(followp["messages"][-1].content)

1. Record video
2. Edit video
3. Upload video


Deep Agent Backends - Where Your Files Actually Live

Every Deep Agent works with a virtual filesystem: its tools read and write files using paths like/notes/todo.txt. But those paths are an abstraction. The backend decides where that data physically lives and that single choice changes everything about persistence, sharing, and durability.

The agent code stays identical across all three. Only the backend changes.

1. StateBackend (default)

Files live in the agent's LangGraph state-i.e. in RAM, tied to a single thread. They're available during the run via result["files"], and you must manually carry that state forward to keep using them.

Where: in-memory dict, inside one thread's state

Survives: within a single conversation/thread only

Gone when: the thread ends or the process exits

Use it for: ephemeral scratch space, temporary working files

2. FilesystemBackend
Files are written to your real disk, under root_dir. With virtual_mode=True, a virtual path like /notes/todo.txt maps to root_dir/notes/todo.txt. These are genuine files you can open in an editor or cat from a terminal.

Where: your actual filesystem, relative to root_dir

Survives across process restarts - it's a real file

Use it for: when the agent should edit real project files

▲ Caution: grants the agent real read/write disk access-point it at a scratch directory, not anything important

3. StoreBackend

Files live in a LangGraph store, scoped by a namespace. This is the only backend whose files persist across threads/conversations-write on one thread, read back on another. With InMemoryStore this works within a single process; swap in a persistent store (e.g. Postgres-backed)for true cross-restart durability.

Where: inside the store object, under a namespace key

Survives: across threads sharing the same store

Gone when the process exits (if using InMemoryStore) - use a persistent store to outlive restarts

* Use it for: long-term memory, per-user file spaces


## Quick comparison

| Backend | Lives in | Cross-thread? | Survives restart? | Real file on disk? |
| --- | --- | --- | --- | --- |
| **StateBackend** | LangGraph state | ❌ | ❌ | ❌ |
| **FilesystemBackend** | Your disk | 🟩 | 🟩 | 🟩 |
| **StoreBackend** | A LangGraph store | 🟩 | only w/ persistent store | ❌ |

> **Mental model:** the agent never knows the difference. Its tools always speak in file paths — the backend quietly translates every read/write into the right operation underneath. That's what lets you swap backends without touching a line of agent logic.
